# P5-5 · GĐ 5 · Fine-tune LoRA

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base.requires_grad_(False)                       # đóng băng W
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))     # B = 0: ban đầu không đổi gì
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

layer = LoRALinear(nn.Linear(320, 320), r=4, alpha=8)
x = torch.randn(3, 320)
print("tham số được train:", sum(p.numel() for p in layer.parameters() if p.requires_grad))   # 4·(320+320) = 2560
print("lúc đầu giống hệt lớp gốc:", torch.allclose(layer(x), layer.base(x)))               # vì B = 0

tham số được train: 2560
lúc đầu giống hệt lớp gốc: True


## Bài tập 5: Tự viết lớp LoRA {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.scale = base, alpha / r
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.merged = False

    def forward(self, x):
        out = self.base(x)
        return out if self.merged else out + self.scale * (x @ self.A.T @ self.B.T)

    @torch.no_grad()
    def merge(self):
        self.base.weight += self.scale * self.B @ self.A
        self.merged = True

base = nn.Linear(768, 768)
x = torch.randn(4, 768)
lora = LoRALinear(base)
print("lúc đầu bằng gốc:", torch.allclose(lora(x), base(x)))
with torch.no_grad():
    lora.B.normal_(0, 0.01)                    # giả lập đã train
before = lora(x)
lora.merge()
print("sau merge không đổi:", torch.allclose(before, lora(x), atol=1e-5))
print("tham số train:", sum(p.numel() for p in lora.parameters() if p.requires_grad))

lúc đầu bằng gốc: True
sau merge không đổi: True
tham số train: 12288
